# Cohort Data Creation

In [ ]:
import pandas as pd
from LabData.DataLoaders.GutMBLoader import GutMBLoader
from LabData.DataLoaders.SubjectLoader import SubjectLoader
from LabData.DataLoaders.DietLoggingLoader import DietLoggingLoader
from LabData.DataLoaders.LifeStyleLoader import LifeStyleLoader
from LabData.DataLoaders.BodyMeasuresLoader import BodyMeasuresLoader
from LabData.DataAnalyses.TenK_Trajectories.utils import get_diet_logging_around_stage
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


In [ ]:
home_path = '/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/'
SPECIES = 'segal_species' # 'segal_species' or 'mpa_species'

# Study configuration
study_ids = [15]  # [15] for Australian cohort, ['AUS'] for AUS
# Map study_ids to study name for file naming
study_name_map = {15: 'AUS', 'AUS': 'AUS'}
study_name = study_name_map.get(study_ids[0] if isinstance(study_ids[0], int) else study_ids[0], f'study_{study_ids[0]}')

min_col_present_frac = 0.05

In [ ]:
diet_mb_10k = pd.read_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")
with open(home_path + f'data/{SPECIES}/my_lists.pkl', 'rb') as file:
    loaded_lists = pickle.load(file)
base_features_10k, all_features_10k, targets_10k = loaded_lists
with open(home_path + f'data/{SPECIES}/scaler.pkl', 'rb') as scaler_file:
        scaler_10k = pickle.load(scaler_file)
diet_mb_10k

In [ ]:
def explore_columns(df):
    for column in df.columns:
        print(column)
        print(df[column].value_counts())

# study_ids is defined in cell 2
subjects_dl = SubjectLoader()
subjects_data = subjects_dl.get_data(groupby_reg='first', study_ids=study_ids)
subjects_df = subjects_data.df
print(subjects_df)

## Load Microbiome Data

In [ ]:
gut_bacteria = GutMBLoader().get_data(SPECIES, subjects_df=subjects_df, study_ids=study_ids,
                                groupby_reg='first', 
                                genotek_vals=[1], min_col_val=1e-4, take_log=True)
gut_bacteria_df = gut_bacteria.df.dropna(axis=1, how='all')
gut_bacteria_df.columns = gut_bacteria_df.columns.str.replace('s__', '')
# with open(home_path + f'data/{species}/my_lists.pkl', 'rb') as file:
#         loaded_lists = pickle.load(file)
# base_features, all_diet_features, targets = loaded_lists
gut_bacteria_df = gut_bacteria_df[targets_10k]



gut_bacteria_df.head(3)

In [ ]:
gut_bacteria_df.shape

In [ ]:
gut_bacteria_df

In [ ]:
gut_bacteria.df_metadata

In [ ]:
gut_bacteria_df = gut_bacteria_df.join(gut_bacteria.df_metadata[['RegistrationCode', 'Date']]).set_index(['RegistrationCode', 'Date'])
gut_bacteria_df.tail(20)

In [ ]:
gut_bacteria_df_date = gut_bacteria_df.copy()

In [ ]:
# Keep only baseline data (first test per person)
gut_bacteria_df = gut_bacteria_df.groupby(level=0).first().sort_index()

gut_bacteria_df

In [ ]:
# Normalize by row.

# Step 1: Convert to normal scale
gut_bacteria_df_normal = 10 ** gut_bacteria_df

# Step 2: Mask of values that are NOT 0.0001
mask = gut_bacteria_df_normal != 0.0001

# Step 3: Row-wise sum of the non-0.0001 values
non_floor_sum = gut_bacteria_df_normal.where(mask).sum(axis=1)

# Step 4: Normalize ONLY the non-0.0001 values, keep 0.0001 unchanged
gut_bacteria_df_normal = gut_bacteria_df_normal.where(~mask, gut_bacteria_df_normal.div(non_floor_sum, axis=0))

# Step 5: Convert back to log10
gut_bacteria_df_log = np.log10(gut_bacteria_df_normal)
gut_bacteria_df = gut_bacteria_df_log
gut_bacteria_df

In [ ]:
# Keep only baseline microbiome data
baseline_mb = gut_bacteria_df#.reset_index(level=[1], drop=True)

In [ ]:
gut_bacteria_df_col = gut_bacteria.df_columns_metadata
# View 5 most common bacteria
gut_bacteria_df_col[gut_bacteria_df_col['Unnamed: 0'].isin(["Rep_485", "Rep_609", "Rep_477", "Rep_449", "Rep_231"])]

In [ ]:
gut_bacteria_df_col.to_pickle(home_path + f"data/mb_names_{study_name.lower()}.pkl")

In [ ]:
gut_bacteria_df_meta = gut_bacteria.df_metadata
print(gut_bacteria_df_meta.head())
# Check that there's only one mb test per person
gut_bacteria_df_meta.RegistrationCode.value_counts()

### Alpha diversity targets

In [ ]:
# Richness
def richness(row):
    filtered = row[row > -4]
    return len(filtered)

baseline_mb['Richness'] = baseline_mb.apply(richness, axis=1)
baseline_mb

In [ ]:
# Shannon Diversity
def shannon(row):
    filtered = row[row > -4]
    filtered = filtered.drop("Richness")
    
    rel_abundance = 10 ** filtered
    ln_rel_abundance = np.log(rel_abundance.replace(0, 1))
    product = rel_abundance * ln_rel_abundance
    ans = -1 * product.sum()
    return round(float(ans), 2)

baseline_mb['Shannon_diversity'] = baseline_mb.apply(shannon, axis=1)
baseline_mb

In [ ]:

# if species == 'mpa_species':
#     path_to_tree = '/net/mraid20/export/genie/Bin/frcfrc/segata.k21.2023-01-31.tree'
# elif species == 'segal_species':
path_to_tree = '/net/mraid20/export/genie/Bin/frcfrc/segal.k21.2023-01-10.tree'
# else:
#     raise ValueError(f"Unknown SPECIES: {species}")

with open(path_to_tree, 'r') as f:
    print(f.read(100))

In [ ]:
import pandas as pd
from skbio import TreeNode

# 1. Load the tree (if not already loaded)
tree = TreeNode.read(path_to_tree)

# 2. Create the Mapping Dictionary
# The user specified: 
#   - Key (Source): First column of dataframe (values like "Rep_33")
#   - Value (Target): The index of the dataframe (values like "fBin__14|gBin__27|sBin__33")
# We create a dictionary: {'Rep_33': 'fBin__14|gBin__27|sBin__33', ...}
name_mapping = pd.Series(
    gut_bacteria_df_col.index.values,       # The Target (Index)
    index=gut_bacteria_df_col.iloc[:, 0]    # The Source (First Column)
).to_dict()

# 3. Rename the Tree Tips
count_renamed = 0
for tip in tree.tips():
    # The current tip name is like "Rep_33.fa.gz"
    # We strip ".fa.gz" to get "Rep_33" so we can look it up in the dictionary
    clean_name = tip.name.replace(".fa.gz", "")
    
    # Check if this Rep ID exists in our mapping
    if clean_name in name_mapping:
        # Update the tip name to the full species string
        tip.name = name_mapping[clean_name]
        count_renamed += 1

print(f"Successfully renamed {count_renamed} tips.")

# Optional: Verify a few tips to make sure it worked
print("First 5 new tip names:")
for tip in list(tree.tips())[:5]:
    print(tip.name)

# 4. Now you can calculate Faith's PD using the dataframe with species index
# Ensure your abundance table columns also match these new tree tip names!

In [ ]:
# Iterate over all nodes (tips and internal nodes)
for node in tree.traverse():
    # If branch length is missing (None), set it to 0.0
    if node.length is None:
        node.length = 0.0

print("Tree patched: All missing branch lengths set to 0.0.")

In [ ]:
import pandas as pd
from skbio.diversity.alpha import faith_pd
import numpy as np

def get_faith_index(df, tree):
    """
    Calculates Faith's Phylogenetic Diversity.
    Fixes the 'Integer Truncation' bug by passing 1s instead of floats.
    """
    # 1. Drop metadata
    taxa_df = df.drop(columns=["Richness", "Shannon_diversity", "Faith_index"], errors='ignore')
    
    # 2. Alignment Check
    tree_tips = {tip.name for tip in tree.tips()}
    valid_cols = [c for c in taxa_df.columns if c in tree_tips]
    taxa_df = taxa_df[valid_cols]

    def calculate_row_pd(row):
        # 3. Filter for Presence
        # Keep species with log abundance > -4
        present_taxa = row[row > -4]
        
        if present_taxa.empty:
            return 0.0
        
        # 4. The Fix: Create a Fake Count Vector of Integers
        # We create a Series of 1s, indexed by the species names.
        # This tells faith_pd: "These species are definitely present."
        present_counts = pd.Series(1, index=present_taxa.index, dtype=int)
        
        try:
            return faith_pd(present_counts, present_taxa.index, tree)
        except ValueError:
            return float('nan')

    # Apply to every row
    return taxa_df.apply(calculate_row_pd, axis=1)

# Usage
gut_bacteria_df['Faith_index'] = get_faith_index(gut_bacteria_df, tree)

# Check results (Should now be > 46.6)
print(gut_bacteria_df[['Faith_index']].head())

In [ ]:
# Sanity check
import matplotlib.pyplot as plt
import scipy.stats as stats

# 1. Calculate Simple Richness (Count of species > -4 log abundance)
# We drop the metadata columns first to ensure we only count bacteria

# 2. Plot
plt.figure(figsize=(8, 6))
plt.scatter(gut_bacteria_df['Richness'], gut_bacteria_df['Faith_index'], alpha=0.5)
plt.xlabel('Species Richness (Count > -4)')
plt.ylabel("Faith's Phylogenetic Diversity")
plt.title('Sanity Check: PD vs Richness')

# 3. Correlation Score
# corr, p_val = stats.spearmanr(gut_bacteria_df['Richness'], gut_bacteria_df['Faith_index'])
# print(f"Spearman Correlation: {corr:.2f} (Should be > 0.8)")
corr, p_val = stats.pearsonr(gut_bacteria_df['Richness'], gut_bacteria_df['Faith_index'])
print(f"Pearson Correlation: {corr:.2f} (Should be > 0.8)")
corr, p_val = stats.pearsonr(gut_bacteria_df['Shannon_diversity'], gut_bacteria_df['Faith_index'])
print(f"Pearson Correlation: {corr:.2f} (Should be > 0.8)")
corr, p_val = stats.pearsonr(gut_bacteria_df['Richness'], gut_bacteria_df['Shannon_diversity'])
print(f"Pearson Correlation: {corr:.2f} (Should be > 0.8)")
plt.show()

In [ ]:
diversity_targets = ['Richness', 'Shannon_diversity', 'Faith_index']

## Load Diet Data

In [ ]:
with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/david_colab/my_lists_diet.pkl', 'rb') as file:
            loaded_lists = pickle.load(file)
base_features, all_diet_features = loaded_lists

In [ ]:
with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/food_shortnames.pkl', 'rb') as file:
    food_shortnames = pickle.load(file)
food_shortnames

In [ ]:
# dll = DietLoggingLoader()
# dlld = dll.get_data(study_ids=15, stage='baseline')
# # log = get_diet_logging_around_stage(dlld.df, delta_before=2, delta_after=14)
# log = dlld.df.reset_index()
# log = log.set_index(['RegistrationCode','Date','food_id'])
# print(log.head(10))
# log.shape

In [ ]:
# log_date = dll.add_new_nutrients(log)
# log_date

In [ ]:
diet_aus = pd.read_csv('/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/nastya/diet_predictions/nutr_full_clean_aus.csv', index_col=0)
diet_aus

In [ ]:
nutr_list_aus = list(diet_aus.columns)

In [ ]:
with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/nutr_list_aus.pkl', 'wb') as file:
    pickle.dump(nutr_list_aus, file)

In [ ]:
nutr_list_no_energy = [nutrient for nutrient in nutr_list_aus if nutrient != "Energy"]
diet_aus[nutr_list_no_energy] = diet_aus[nutr_list_no_energy].div(diet_aus['Energy'], axis=0)

In [ ]:
diet_aus

In [ ]:
# Unite similar features by summing them up
# diet_aus['omega_6'] = diet_aus[['PUFA 18:2 n-6 c,c', 'PUFA 20:4 n-6', 'PUFA 18:2 CLAs', 'PUFA 18:2 i', 'PUFA 20:2 n-6 c,c', 'PUFA 2:4 n-6', 'PUFA 22:4', 'PUFA 18:3 n-6 c,c,c']].sum(axis=1)
# diet_aus['omega_3'] = diet_aus[['PUFA 18:3 n-3 c,c,c (ALA)', 'PUFA 18:4', 'PUFA 20:5 n-3 (EPA)', 'PUFA 22:6 n-3 (DHA)', 'PUFA 22:5 n-3 (DPA)', 'PUFA 20:3 n-3']].sum(axis=1)
# diet_aus['vitamin_E'] = diet_aus[['Vitamin E (alpha-tocopherol)', 'Tocopherol, beta', 'Tocopherol, delta', 'Tocopherol, gamma', 'Tocotrienol, alpha', 'Tocotrienol, beta', 'Tocotrienol, delta', 'Tocotrienol, gamma']].sum(axis=1)

# Drop the original columns after uniting
# columns_to_drop = [
#     'SFA 4:0', 'SFA 6:0', 'SFA 8:0', 'SFA 10:0', 'SFA 12:0', 'SFA 14:0', 'SFA 16:0', 'SFA 18:0', 'SFA 13:0', 'SFA 15:0', 'SFA 17:0', 'SFA 20:0', 'SFA 22:0', 'SFA 24:0',
#     'MUFA 14:1', 'MUFA 15:1', 'MUFA 16:1', 'MUFA 17:1', 'MUFA 18:1', 'MUFA 18:1 c', 'MUFA 20:1', 'MUFA 22:1', 'MUFA 24:1 c', 'MUFA 16:1 c', 'MUFA 22:1 c', 'MUFA 18:1-11 t (18:1t n-7)',
#     'PUFA 18:2 n-6 c,c', 'PUFA 20:4 n-6', 'PUFA 18:2 CLAs', 'PUFA 18:2 i', 'PUFA 20:2 n-6 c,c', 'PUFA 2:4 n-6', 'PUFA 22:4', 'PUFA 18:3 n-6 c,c,c',
#     'PUFA 18:3 n-3 c,c,c (ALA)', 'PUFA 18:4', 'PUFA 20:5 n-3 (EPA)', 'PUFA 22:6 n-3 (DHA)', 'PUFA 22:5 n-3 (DPA)', 'PUFA 20:3 n-3',
#     'TFA 16:1 t', 'TFA 18:1 t', 'TFA 18:2 t not further defined', 'TFA 18:2 t,t', 'TFA 22:1 t',
#     'Vitamin A, IU', 'Carotene, beta', 'Retinol', 'Carotene, alpha', 'Cryptoxanthin, beta',
#     'Vitamin D2 (ergocalciferol)', 'Vitamin D3 (cholecalciferol)', 'Vitamin D (D2 + D3), International Units', 
#     'Vitamin K (Dihydrophylloquinone)', 'Vitamin K (Menaquinone-4)', 'Vitamin K (phylloquinone)',
#     'Vitamin E (alpha-tocopherol)', 'Tocopherol, beta', 'Tocopherol, delta', 'Tocopherol, gamma', 'Tocotrienol, alpha', 'Tocotrienol, beta', 'Tocotrienol, delta', 'Tocotrienol, gamma', 'Vitamin E, added',
#     'Folate, DFE', 'Sugars, total including NLEA',
#     'Fatty acids, total trans-monoenoic', 'Fatty acids, total trans-polyenoic', 'Folate, DFE', 'Folate, food', 'Folic acid',
#     'Galactose', 'Lactose', 'Maltose',  'PUFA 18:2', 'PUFA 18:3', 'PUFA 18:3i', 'PUFA 20:3', 'PUFA 20:4', 'PUFA 21:5',
#     'Theobromine', 'Stigmasterol', 'Beta-sitosterol', 'Sucrose', 'matched_food_score',
#     'Isoleucine', 'Leucine', 'Valine', 'Lysine', 'Threonine', 'Methionine', 'Phenylalanine', 'Tryptophan', 'Histidine',
#     'Tyrosine', 'Arginine', 'Cystine', 'Serine', 'Alanine', 'Aspartic acid', 'Glutamic acid', 'Glycine', 'Hydroxyproline', 'Proline'
# ]

# diet_aus.drop(columns=columns_to_drop, inplace=True)

# # Optional: remove additional duplicates like 'Vitamin B-12, added' if needed
# diet_aus.drop(columns=['Vitamin B-12, added'], inplace=True, errors='ignore')

# # Ensure to keep only 'Folate, total'
# diet_aus = diet_aus.loc[:, ~diet_aus.columns.duplicated()]


### END

### BMI

In [ ]:
bml = BodyMeasuresLoader()
# bmld = bml.get_data(study_ids=study_ids, cols=['weight', 'height', 'bmr'])
bmld = bml.get_data(study_ids=study_ids)
bmldf = bmld.df
bmldf.info()

In [ ]:
# if stage == 'baseline':
    # Keep only baseline
bmldf = bmldf[~bmldf.index.get_level_values(0).duplicated()]
bmldf = bmldf.reset_index(level=[1], drop=True)
# elif stage == '02_00_visit':
#     # Keep only the second entry (2nd visit)
#     bmldf = bmldf.groupby(level=0).nth(1) 
# elif stage == '04_00_visit':
#     # Keep only the second entry (3rd visit)
#     bmldf = bmldf.groupby(level=0).nth(1) 
bmldf

## Combine Dataframes

In [ ]:
# baseline_nutrients = baseline_nutrients.set_index('RegistrationCode')
# baseline_nutrients.columns = [rename_dict.get(item, item) for item in baseline_nutrients.columns]
# baseline_nutrients

In [ ]:
# diet_mb = baseline_foods.join(baseline_nutrients, how='inner')
# # diet_mb = diet_mb.dropna()
# diet_mb

In [ ]:
subjects_df = subjects_df.reset_index(level=[1], drop=True)
subjects_df

In [ ]:
base_features = ["age", "gender"]
subjects_df.index = subjects_df.index.astype('int')
diet_aus = diet_aus.join(subjects_df[base_features])
diet_aus = diet_aus.rename(columns={'gender': 'sex'})
# diet_mb = diet_mb.reset_index(level=[1], drop=True)
# diet_mb = diet_mb.dropna()
diet_aus

In [ ]:
bmldf.index = bmldf.index.astype('int')

diet_aus = diet_aus.join(bmldf['bmi'], how='inner')
diet_aus.shape

In [ ]:
baseline_mb.index = baseline_mb.index.astype('int')
diet_mb_baseline = diet_aus.join(baseline_mb)
diet_mb_baseline

In [ ]:
# Baseline data shape
print(diet_mb_baseline.shape)

In [ ]:
# from sklearn.preprocessing import StandardScaler

# with open(home_path + f'data/{SPECIES}/age_scaler.pkl', 'rb') as f:
#     age_scaler = pickle.load(f)

# with open(home_path + f'data/{SPECIES}/mb_scaler.pkl', 'rb') as f:
#     mb_scaler = pickle.load(f)

# with open(home_path + f'data/{SPECIES}/div_scaler.pkl', 'rb') as f:
#     div_scaler = pickle.load(f)

# diet_scaler = StandardScaler()
# diet_mb_10k_scaled = diet_scaler.fit_transform(diet_mb_10k[aus_10k_shared_features])

# # # Apply the scaler to the dataframe
# diet_mb_baseline.loc[:, aus_10k_shared_features] = diet_scaler.transform(diet_mb_baseline[aus_10k_shared_features])
# diet_mb_baseline.loc[:, ["age"]] = age_scaler.transform(diet_mb_baseline[["age"]])
# diet_mb_baseline.loc[:, targets_10k] = mb_scaler.transform(diet_mb_baseline[targets_10k])
# diet_mb_baseline.loc[:, ["Richness", "Shannon_diversity"]] = div_scaler.transform(diet_mb_baseline[["Richness", "Shannon_diversity"]])
# # diet_mb_baseline[aus_10k_shared_features] = pd.DataFrame(scaler.transform(diet_mb_baseline[aus_10k_shared_features]), columns=diet_mb_baseline[aus_10k_shared_features].columns, index=diet_mb_baseline[aus_10k_shared_features].index)
# diet_mb_baseline.describe()

In [ ]:
# Scaling code for baseline (commented out)
# diet_mb_baseline.loc[:, aus_10k_shared_features] = diet_scaler.transform(diet_mb_baseline[aus_10k_shared_features])
# diet_mb_baseline.loc[:, ["age"]] = age_scaler.transform(diet_mb_baseline[["age"]])
# diet_mb_baseline.loc[:, targets_10k] = mb_scaler.transform(diet_mb_baseline[targets_10k])
# diet_mb_baseline.loc[:, ["Richness", "Shannon_diversity"]] = div_scaler.transform(diet_mb_baseline[["Richness", "Shannon_diversity"]])
# diet_mb_baseline.describe()

In [ ]:
# Count rows that have at least one NaN
nan_rows = diet_mb_baseline[diet_mb_baseline.isna().any(axis=1)]
num_nan_rows = len(nan_rows)

# Find which columns contain NaN values
nan_features = diet_mb_baseline.columns[diet_mb_baseline.isna().any()].tolist()

print(f"Number of rows with at least one NaN: {num_nan_rows}")
print("Features that contain NaN values:")
print(nan_features)


In [ ]:
print("\nNaN count per column:")
print(diet_mb_baseline.isna().sum()[diet_mb_baseline.isna().sum() > 0])
diet_mb_baseline = diet_mb_baseline.dropna(how='any')


In [ ]:
print(diet_mb_baseline.shape)

In [ ]:
diet_mb_baseline.to_pickle(home_path + f'data/segal_species/diet_mb_{study_name.lower()}_baseline.pkl')
with open(home_path + f'data/segal_species/my_lists_{study_name.lower()}.pkl', 'wb') as file:
    pickle.dump([nutr_list_aus, targets_10k], file)

In [ ]:
diet_mb_baseline.columns

In [ ]:
print(len(nutr_list_aus))
print(len(targets_10k))

### Phenotype data

In [ ]:
all_meas=pd.read_csv('/net/mraid20/export/genie/LabData/Data/StudySpecificData/AUS_Predict/intervention/all_results.csv')
all_meas=all_meas[[col for col in all_meas.columns if 'start' in col]]
all_meas=all_meas.rename(columns={col:col.replace('_start', '') for col in all_meas.columns})